# Exoplanet Detection - RV Method

Adapted from lab exercise written by Prof. Jenny Greene at Princeton University

## Introduction

51 Pegasus b was the first exoplanet to be discovered orbiting a main-sequence star. That star, the Sun-like 51 Pegasi (with a stellar mass $m_{\star}=1.12\pm0.06 M_{\odot}$), lies $\approx$51 light-years away in the Pegasus constellation. 51 Peg b was found by Mayor & Queloz (1995) using the radial velocity (RV) method, which relies on an ability to measure small Doppler shifts in the host star’s spectral lines as the star wobbles due to its orbiting planet’s periodic gravitational pull. The term “radial velocity” refers specifically to the component of the star’s velocity oriented along the observer’s line-of-sight. Once the radial velocity $v_r$ is measured as a function of time $t$, its periodicity tells you the orbital period $P$ of the planet, which, if the orbit is indeed Keplerian, can give you the semi-major axis $a$ via

$$P^2 = \frac{(2\pi)^2}{GM} a^3 $$

The semi-amplitude of the signal $K$ is related by Kepler’s laws to the masses of the two bodies and the orbital eccentricity $e$ via:

$$K = \left(\frac{2\pi G}{P}\right)^{1/3} \frac{m_p {\rm sin}i}{m_{\star}^{2/3}} \frac{1}{\sqrt{1-e^2}}$$

If the stellar mass $m_{\star}$ (assumed here to be $\gg m_p$, the planet mass) and the inclination of the orbital plane with respect to the line of sight $i$ are known, then the planet mass $m_p$ can be derived from from expression above once you know the eccentricity e. (Note that, if $i = 0$, there is no component of the orbital velocity along the line of sight, and so $K = 0$. Thus, $m_p {\rm sin} i$ is a _lower limit_ on $m_p$).

But how does one know the eccentricity? For that, you need a fit to the radial velocity curve ($v_r(t)$), which we can do using a least-squares method.

You will use RV data to discover 51 Peg b for yourself and use a least-squares fit to constrain its orbital parameters.

## Import Packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.stats import LombScargle
from scipy.optimize import curve_fit, newton
from astropy import units as u
import math

%matplotlib inline

Download the data file. There are three columns of data in this file—Heliocentric Julian Date (HJD, measured in days), radial velocity (relative to the barycenter, measured in m/s), and uncertainty (measured in m/s).

In [2]:
# If you open the data file, you see comments at the top about the file -- we don't need this in our analysis
    # We determine the number of rows that contain info we don't want
other_info = 18 # number of rows of comments
n_header = 4 # number of rows taken up by headers

# Read in the data
    # Hint: use 'other_info' & 'n_header' when reading in the data! (look at documentation for the function you use)
data = 

SyntaxError: invalid syntax (<ipython-input-2-7c7fe915f489>, line 1)

In [ ]:
data

In [ ]:
# Column names are shown in the file. Save each column into an array
    # Tip: write descriptive comments to remember what your variables represent

# (insert code)

In [ ]:
# Plot the data + errors

# (insert code)

The points don't look like the curve we expect (see the image for the radial velocity method in the "Detection" portion of NAAP). This is because of uneven sampling (in time). We use ``LombScargle`` to extract the period. Here is the documentation: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.lombscargle.html 

In [ ]:
# Edit the code below
    # Note: using 3 quotations '''Text''' is another way to comment out code! Commonly used for blocks of code:
'''
    This block of text
    will not run
    (random text)
'''
    # Also see "Better function writing" section near the end of Python for Astronomers tutorial Part 2
frequency, power = LombScargle('''Insert array of Julian dates''','''Insert array of radial velocities''','''Insert array of uncertainties''').autopower()

# Plot Power vs. Period
    # Remember: period = 1/frequency
    # Use a logarithmic scale for the x-axis

# (insert code)

Determine the most likely orbital period (this would be the period that shows the highest power in the plot above). *Hint: use ``np.argmax``*

In [ ]:
# Find the period peak of the above plot
period = 
print("The fit period is {:.5f} days".format(period))

``LombScargle`` can calculate a model to fit to the data, given an time array and the orbital period of the planet. From this we can obtain a good estimate for the RV semi-amplitude $K$. (https://docs.astropy.org/en/latest/timeseries/lombscargle.html has examples under "The Lomb-Scargle Model" section).

In [ ]:
# Calculate model fit based on period determined above
tfit = np.linspace(0, period, 1000)
rvfit = LombScargle('''Insert array of Julian dates''','''Insert array of radial velocities''','''Insert array of uncertainties''').model(tfit,1/period)

# Compute the semi-amplitude K from the model fit
K = 0.5*(np.max(rvfit)-np.min(rvfit))
print("The fit semi-amplitude is {:.4f} m/s".format(K))

# Phase the data and plot it (+errors) with the fit
phased_data = '''Insert array of Julian dates''' % period

plt.plot(tfit,rvfit,'-r',lw=3,label='LS model') # this is code to plot the fit

# (insert code)

Using the estimates determined above, we can find better parameter values (along with other values of interest) by fitting the data to the radial velocity equation: <br>
$v_{r, {\rm fit}}(t) = v_{r,0} + K\left[{\rm cos}(\theta(t) + \omega) + e{\rm cos}\omega \right]$ <br>
where $v_{r0}$ is a constant velocity offset, $\omega$ is a constant related to the phase of the orbit (called the "argument of periastron"), and $\theta$ is called the "true anomaly." The time at which planet reaches $\theta=0$ is defined as $\tau$. If you're interested, this document outlines the derivation for this formula: https://sites.astro.caltech.edu/~srk/BlackHoles/Literature/RV_Derivation.pdf (pg. 22).

In [ ]:
# Defining the function to fit to the data
def keplerian_fit(t, K, P, e, w, tau, vr0):
    def f(x):
        return x - e*np.sin(x) - (t-tau)*2*np.pi/P
    
    e_anomaly = newton(f,(t-tau)*2*np.pi/P)
                                
    theta = 2*np.arctan2(np.sqrt(1.+e)*np.sin(0.5*e_anomaly),
                         np.sqrt(1.-e)*np.cos(0.5*e_anomaly))
                                
    return K*(np.cos(theta+w)+e*np.cos(w))+vr0

# Finding best fit parameters
guess = [--,--,0.1,1,1,0] # input "initial guesses" for the parameters (K, P, e, w, tau, vr0) of the system
popt,pcov = curve_fit(--,--,--,sigma=er,absolute_sigma=True,p0=guess)
(K,P,e,w,tau,vr0) = popt
(K_er,P_er,e_er,w_er,tau_er,vr0_er) = np.sqrt(np.diag(pcov))


print('Best fit parameters:')
print('   K = {:.3f} +/- {:.3f} m/s'.format(K, K_er))
print('   P = {:.7f} +/- {:.7f} d'.format(P, P_er))
print('   e = {:.3f} +/- {:.3f}'.format(e, e_er))

# Phase the data and plot it (+errors) with the fit

# (insert code)


Calculate the semi-major axis $a$ (and uncertainties) of the planet's orbit, using Kepler's third law. Give your answer in units of AU. How does this planet's orbit compare to the orbit of Mercury?

Now solve for $m_p {\rm sin}i$ with uncertainties. Report your final answer in terms of _Jupiter's_ mass ($M_J$). How does 51 Peg b's mass compare to the mass of Jupiter?